In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import sys
import time

# Analysis of the results of the CSV'S resulting from the sim

In this notebook it will be shown the process of analyzing the resulting sim data and stablishing a pipeline which will lead to a .py projectwhich will automatically run after the GAMA sim and be able to extract the results via graphics and similar

First there is a lot of csvs. we need to joint them all into one

In [14]:
BASE_SIM_DIR = "20260312_143150"
#HARDCODDED NUMBER OF SIMS

N_SIMS = 1000

columns = ["simulation_id", "bus_code", "stop_num","timeslot","boards","residual_boards","dropoffs", "occupation", "time_between_stops"]

joint_dataframe_results = pd.DataFrame(columns=columns)

dataframes_to_join_results = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/RESULTS/log_Simulation {iteration}.csv", encoding="utf-8")
    dataframes_to_join_results.append(new_dataframe)



joint_dataframe_results = pd.concat(data for data in dataframes_to_join_results)
joint_dataframe_results = joint_dataframe_results.rename(columns={"stops_listself.directionself.stops_reached.stopNum":"stop_num", "boards - residual_boards":"boards","self.passengers":"passengers"})

In [15]:
N_SIMS = 1000


columns = ["simulation_id", "bus_code","travels_completed", "stop_num", "timeslot", "event_code","time_event","param_event_1", "param_event_2"]

joint_dataframe_events = pd.DataFrame(columns=columns)

dataframes_to_join_events = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/EVENTS/log_Simulation {iteration}.csv", encoding="utf-8")
    dataframes_to_join_events.append(new_dataframe)



joint_dataframe_events = pd.concat(data for data in dataframes_to_join_events)
joint_dataframe_events = joint_dataframe_events.rename(columns={"stops_listself.directionself.stops_reached.stopNum":"stop_num","'DELAY'": "event_type"})

Once joined the data, its time to start making the graphics results andif well done, elevate the process to a .py code that accepts the ISODATE of the sim as a parameter, and build a directory which holds the graphics results.

## Graphic 1. Statistics of simulation Occupation VS REAL TIME occupation

In [16]:
# la idea de esto es generar gráficas a mansalva. La idea detrás ed esto es poder tener tantas gráficas como se guste y en la interfaz de usuario poder agrupar esas gráficas para que se den varios puntos de vista.


#idea 1. A parte de hacer gráficas para todos los timeslots, agrupar por hardcodded timestamps (Mañana[5], Hora punta[6,9], Valle de Mañana[10,13], Hora tarde[14,19], noche[20,23]
timeslots = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]



In [17]:
joint_dataframe_results

,world.name,self.id,self.travels_completed,stop_num,self.direction,current_time,TimeSlot,boards,residual_boards,dropoffs,passengers,self.time_travelled
0,Simulation 0,3560,0,2646,toA,6:00,6,0,0,0,0,0.0
1,Simulation 0,3560,0,562,toA,6:02,6,36,0,0,36,140.0
2,Simulation 0,3560,0,4090,toA,6:05,6,14,0,0,50,140.0
3,Simulation 0,3560,0,4188,toA,6:08,6,5,0,0,55,170.0
4,Simulation 0,3560,0,5939,toA,6:11,6,3,0,0,58,150.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1468,Simulation 999,3553,8,1929,toA,23:39,23,8,0,1,49,190.0
1469,Simulation 999,3553,8,4032,toA,23:41,23,3,0,0,52,70.0
1470,Simulation 999,3553,8,5360,toA,23:43,23,5,0,5,52,80.0
1471,Simulation 999,3553,8,82,toA,23:46,23,3,0,14,41,160.0


In [18]:
joint_dataframe_events

,world.name,self.id,self.travels_completed,stop_num,self.direction,event_type,current_time,self.departure_time,(world.time_to_minutes(t:current_time)) - (world.time_to_minutes(t:self.departure_time))
0,Simulation 0,3569,1,5727,toB,DELAY,6:59,6:55,4
1,Simulation 0,3580,1,5727,toB,DELAY,7:56,7:55,1
2,Simulation 0,3577,1,2646,toA,DELAY,8:26,8:10,16
3,Simulation 0,3578,1,5727,toB,DELAY,8:45,8:40,5
4,Simulation 0,3580,3,5727,toB,DELAY,9:27,9:25,2
...,...,...,...,...,...,...,...,...,...
20,Simulation 999,3551,9,5727,toB,DELAY,20:22,20:20,2
21,Simulation 999,3587,11,5727,toB,DELAY,21:06,21:05,1
22,Simulation 999,3583,12,5727,toB,DELAY,21:39,21:30,9
23,Simulation 999,3583,13,2646,toA,DELAY,22:11,22:10,1


# Estadísticas generales. Medidas de Rendimiento. % de ocupación medio, general, por franja horaria, Enfrentamiento real vs teórico "normal" de ocupacion

In [19]:
#KPI 1. % de ocupación medio. % de ocupacion por franja horaria. % de ocupación de cada autobús en acda una de sus franjas
MAX_CAPACITY = 80

def time_to_minutes(time_str):
    h, m = map(int, time_str.split(":"))
    return h * 60 + m


#La easy.
percentage_occupation_per_stop = 0

#In toA

json_output_general_info = {}



data_toA = joint_dataframe_results[joint_dataframe_results["self.direction"] == "toA"]
data_toB = joint_dataframe_results[joint_dataframe_results["self.direction"] == "toB"]

json_output_general_info["Occupation_toA"] = (data_toA["passengers"].mean()/MAX_CAPACITY)*100
json_output_general_info["Occupation_toB"] = (data_toB["passengers"].mean()/MAX_CAPACITY)*100
json_output_general_info["Occupation"] = (joint_dataframe_results["passengers"].mean()/MAX_CAPACITY)*100



#Empiezan las graficassss
#graficas de evolucion a lo largo del tiempon de valor de %Occupation
list_occupation_values_toA = []
list_occupation_values_toB = []
list_occupation = []
for timeslot in timeslots:
    data_toA_per_ts = data_toA[data_toA["TimeSlot"] == timeslot]
    data_toB_per_ts = data_toB[data_toB["TimeSlot"] == timeslot]
    data_general = joint_dataframe_results[joint_dataframe_results["TimeSlot"] == timeslot]
    list_occupation_values_toA.append((data_toA_per_ts["passengers"].mean()/MAX_CAPACITY)*100)
    list_occupation_values_toB.append((data_toB_per_ts["passengers"].mean()/MAX_CAPACITY)*100)
    list_occupation.append((data_general["passengers"].mean()/MAX_CAPACITY)*100)


plt.style.use('seaborn-v0_8-whitegrid')

def save_barcode(horizontal_data, vertical_data, horizontal_label, vertical_label, leyend, label, filename):
    x = np.arange(len(horizontal_data))
    plt.figure(figsize=(12,5))
    plt.bar(x, vertical_data, width=0.4, label=leyend)
    plt.xticks(x + 0.2, horizontal_data, rotation=90)
    plt.ylabel(vertical_label)
    plt.xlabel(horizontal_label)
    plt.title(label)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


save_barcode(timeslots, list_occupation_values_toA , "TimeSlots", "% of Occupation",  "Direction to A", "Bus occupation per timeslot", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/occupation_per_timeslot_toA.png")

save_barcode(timeslots, list_occupation_values_toB , "TimeSlots", "% of Occupation",  "Direction to B", "Bus occupation per timeslot", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/occupation_per_timeslot_toB.png")

save_barcode(timeslots, list_occupation , "TimeSlots", "% of Occupation",  "General Data", "Bus occupation per timeslot",
             f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/general/occupation_per_timeslot_general.png")


stops_list_toA = data_toA["stop_num"].unique()
stops_list_toB = data_toB["stop_num"].unique()

stops_list_toA = [int(data) for data in stops_list_toA]
stops_list_toB = [int(data) for data in stops_list_toB]


for stopcode in stops_list_toA:
    #obtain the data from that stop and extract per timeslots its value
    list_per_ts_per_stop = []
    data_stopcode = data_toA[data_toA["stop_num"] == stopcode]
    if stopcode == stops_list_toA[-1]:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["TimeSlot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/stops/occupation_per_timeslot_{stopcode}.png")
    else:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["TimeSlot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["passengers"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/stops/occupation_per_timeslot_{stopcode}.png")



for stopcode in stops_list_toB:
    #obtain the data from that stop and extract per timeslots its value
    list_per_ts_per_stop = []
    data_stopcode = data_toB[data_toB["stop_num"] == stopcode]
    if stopcode == stops_list_toB[-1]:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["TimeSlot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/stops/occupation_per_timeslot_{stopcode}.png")
    else:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["TimeSlot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["passengers"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/stops/occupation_per_timeslot_{stopcode}.png")



#Now occupation per physical bus
unique_bus = joint_dataframe_results["self.id"].unique()


input_json_route = f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/includes/INPUTS/input_data_buses.json"

with open(input_json_route, "r") as f:
    input_json = json.load(f)

new_bus_json_data = {}

for bus in unique_bus:

    trip_occupation_percentage = 0

    data_per_bus = joint_dataframe_results[joint_dataframe_results["self.id"] == bus]
    events_per_bus = joint_dataframe_events[joint_dataframe_events["self.id"] == bus]

    unique_trips = data_per_bus['self.travels_completed'].unique()
    unique_trips = [int(data) for data in unique_trips]

    new_bus_json_data[f"{bus}"] = {}

    #Loop over each trip
    for trip in unique_trips:



        events_per_unique_trip =events_per_bus[events_per_bus["self.travels_completed"] == trip]

        total_percentage_of_delays_per_trip = (len(events_per_unique_trip)/N_SIMS)*100




        mean_delay_in_minutes_per_trip = (pd.Series([time_to_minutes(data) for data in events_per_unique_trip["current_time"]]) - pd.Series([time_to_minutes(data) for data in events_per_unique_trip["self.departure_time"]])).mean()



        data_inputs = input_json[str(bus)][trip]

        data_simulation_per_trip = data_per_bus[data_per_bus['self.travels_completed'] == trip]
        trip_occupation_percentage = (data_simulation_per_trip["passengers"].mean()/MAX_CAPACITY)*100

        events_per_trip_per_bus = events_per_bus[events_per_bus['self.travels_completed'] == trip]

        mean_time_of_trip = 0
        #Extract the mean ocupation for saving the plot

        list_mean_values = []
        direction = ""
        if int(data_inputs[0]) == stops_list_toA[0]:
            direction = "to_A"

            #extract the mean estimated time. Calculating the mean arrival time to stop stops_list_toA[-1] and the mean departure time from stops_list_toA[-1]
            data_per_trip_last_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toA[-1]]
            time_per_last_stop = data_per_trip_last_stop["current_time"]
            time_per_last_stop = [time_to_minutes(data) for data in time_per_last_stop]
            data_per_trip_first_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toA[0]]
            time_per_first_stop = data_per_trip_first_stop["current_time"]
            time_per_first_stop = [time_to_minutes(data) for data in time_per_first_stop]
            mean_time_of_trip = sum(time_per_last_stop)/len(time_per_last_stop) - sum(time_per_first_stop)/len(time_per_first_stop)

            for stop in stops_list_toA:
                data_per_trip_per_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stop]
                list_mean_values.append( (data_per_trip_per_stop["passengers"].mean()/MAX_CAPACITY)*100)

            save_barcode(stops_list_toA, list_mean_values , "Stops in toA Direction", "% of Occupation",  f"Occupation for trip", f"Bus occupation per stop for unique bus {bus} with estimate departure {data_inputs[1]}", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/mean_occupation_per_stop_toA_{bus}_{trip}.png")




        else:
            direction = "toB"

            #extract the mean estimated time. Calculating the mean arrival time to stop stops_list_toA[-1] and the mean departure time from stops_list_toA[-1]
            data_per_trip_last_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toB[-1]]
            time_per_last_stop = data_per_trip_last_stop["current_time"]
            time_per_last_stop = [time_to_minutes(data) for data in time_per_last_stop]
            data_per_trip_first_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toB[0]]
            time_per_first_stop = data_per_trip_first_stop["current_time"]
            time_per_first_stop = [time_to_minutes(data) for data in time_per_first_stop]
            mean_time_of_trip = sum(time_per_last_stop)/len(time_per_last_stop) - sum(time_per_first_stop)/len(time_per_first_stop)


            for stop in stops_list_toB:
                data_per_trip_per_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stop]
                list_mean_values.append( (data_per_trip_per_stop["passengers"].mean()/MAX_CAPACITY)*100)

            save_barcode(stops_list_toB, list_mean_values , "Stops in toB Direction", "% of Occupation",  f"Occupation for trip", f"Bus occupation per stop for unique bus {bus} with estimate departure {data_inputs[1]}", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/mean_occupation_per_stop_toB_{bus}_{trip}.png")

        #print(f"for trip {data_inputs[0]}-{data_inputs[1]} for bus {bus} we have a mean time of trip {mean_time_of_trip}")
        #Now data_inputs will act as the key for the data for that specific trip. The data will be. Mean % of occupation. (Mean % of occupation per stop(indicating direction)) Mean %of delays and Mean minutes of delay in case it exists.
        new_bus_json_data[f"{bus}"][f"{data_inputs[0]}-{data_inputs[1]}"] = {"Mean_Occupation_Percentage": trip_occupation_percentage, "Direction": direction, "Mean_time_of_trip": mean_time_of_trip, "Percentage_sims_delay_departure": total_percentage_of_delays_per_trip, "Mean_Minutes_departure_delay" : mean_delay_in_minutes_per_trip}


#Important KPI. Theoretical vs Real Distribution of Occupation per Timeslot per Stop


#Time between stops.

In [21]:
print(new_bus_json_data)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/general_info.json", "w") as f:
    json.dump(new_bus_json_data, f)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/json_output_general_info.json", "w") as f:
    json.dump(json_output_general_info, f)

{'3560': {'2646-6:00': {'Mean_Occupation_Percentage': np.float64(68.52000000000001), 'Direction': 'to_A', 'Mean_time_of_trip': 37.30799999999999, 'Percentage_sims_delay_departure': 0.0, 'Mean_Minutes_departure_delay': nan}, '5727-6:35': {'Mean_Occupation_Percentage': np.float64(49.4396875), 'Direction': 'toB', 'Mean_time_of_trip': 27.557999999999993, 'Percentage_sims_delay_departure': 82.1, 'Mean_Minutes_departure_delay': np.float64(2.943970767356882)}, '2646-7:05': {'Mean_Occupation_Percentage': np.float64(55.217666666666666), 'Direction': 'to_A', 'Mean_time_of_trip': 44.69200000000001, 'Percentage_sims_delay_departure': 44.800000000000004, 'Mean_Minutes_departure_delay': np.float64(2.564732142857143)}, '5727-8:10': {'Mean_Occupation_Percentage': np.float64(24.580729166666664), 'Direction': 'toB', 'Mean_time_of_trip': 30.678999999999974, 'Percentage_sims_delay_departure': 0.0, 'Mean_Minutes_departure_delay': nan}, '2646-8:45': {'Mean_Occupation_Percentage': np.float64(16.4700000000000